# MuJoCo Humanoid — Interactive Demo

This notebook demonstrates GPU-accelerated humanoid simulation using MuJoCo and dm_control.
The humanoid model ships with dm_control — no external files needed.

## Background: Key Concepts for Cloud Developers

**MuJoCo** (Multi-Joint dynamics with Contact) is a fast, accurate physics engine designed for robotics and biomechanics simulation. Originally developed at the University of Washington, it was acquired by Google DeepMind and made open-source in 2022. Think of it as the "compute engine" that calculates how bodies, joints, and forces interact over time.

**MJCF** (MuJoCo Modeling Format) is the XML schema used to describe a simulated robot or scene. An MJCF file defines the robot's bodies (links), joints (degrees of freedom), actuators (motors), and visual/collision geometry. If MuJoCo is the engine, MJCF is the blueprint.

**dm_control** is DeepMind's Python library that wraps MuJoCo with a suite of standard robotics tasks (stand, walk, run, etc.) and pre-built models like the humanoid below. It gives you a clean reinforcement-learning interface — observations in, actions out — so you can focus on training algorithms rather than physics plumbing.

## Rendering the Humanoid

In simulation, **rendering** means converting the current physics state (joint angles, body positions) into a visual image — pixels you can look at. This is separate from the physics computation itself; the simulation runs whether or not you render it.

We use **offscreen (headless) rendering** here because cloud VMs typically have no monitor attached. MuJoCo renders into an in-memory framebuffer, and we display the result inline using `mediapy`. This is the same approach you would use in any Azure VM or container environment.

The cell below loads the humanoid model and renders it in its **default pose** — all joints at zero, standing upright with no forces applied. You will see a 21-body, 28-joint humanoid with 21 actuators (motors).

In [ ]:
import mujoco
import numpy as np
import mediapy
import os
import dm_control.suite as suite

# Load the humanoid model from dm_control
suite_dir = os.path.dirname(suite.__file__)
xml_path = os.path.join(suite_dir, 'humanoid.xml')

model = mujoco.MjModel.from_xml_path(xml_path)
# Override offscreen framebuffer for HD rendering
model.vis.global_.offwidth = 1920
model.vis.global_.offheight = 1080
data = mujoco.MjData(model)

print(f'MuJoCo version: {mujoco.__version__}')
print(f'Model: dm_control humanoid')
print(f'Bodies: {model.nbody}, Joints: {model.njnt}, Actuators: {model.nu}')
print(f'Timestep: {model.opt.timestep}s')

## GPU Acceleration with JAX and MJX

**JAX** is Google's open-source library for high-performance numerical computing. It provides NumPy-like syntax but can compile and run your code on GPUs and TPUs automatically. JAX also supports automatic differentiation, which is essential for modern machine learning.

**MJX** is MuJoCo's JAX backend. It re-implements the MuJoCo physics pipeline in JAX, which means the entire simulation can run on a GPU. The big win: you can run **thousands of simulations in parallel** on a single GPU, dramatically speeding up reinforcement learning training that would otherwise take hours on CPU.

**Why this matters for Azure deployments:** On a GPU-enabled VM (e.g., NC-series), MJX can deliver orders of magnitude more simulation throughput. The cell below checks whether a GPU is available and, if so, benchmarks MJX step performance. If you are running on a CPU-only preset, the GPU check will simply report "not available" — that is perfectly fine for learning and experimentation. The CPU path used in earlier cells works everywhere.

In [ ]:
# Render the humanoid at rest
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, height=480, width=640)
renderer.update_scene(data)
mediapy.show_image(renderer.render())

In [ ]:
# Simulate with sinusoidal control and render video
mujoco.mj_resetData(model, data)
frames = []
duration = 5.0  # seconds
fps = 30
frame_skip = int(1.0 / (fps * model.opt.timestep))

n_steps = int(duration / model.opt.timestep)
for step in range(n_steps):
    t = data.time
    for i in range(model.nu):
        phase = 2.0 * np.pi * (t + i * 0.15)
        data.ctrl[i] = 0.4 * np.sin(phase)
    mujoco.mj_step(model, data)
    if step % frame_skip == 0:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

print(f'Simulated {duration}s, captured {len(frames)} frames')
mediapy.show_video(frames, fps=fps)

In [ ]:
# Check GPU availability for MJX (JAX-accelerated MuJoCo)
import jax
print(f'JAX devices: {jax.devices()}')
print(f'GPU available: {any(d.platform == "gpu" for d in jax.devices())}')

if any(d.platform == 'gpu' for d in jax.devices()):
    from mujoco import mjx
    import time
    
    mjx_model = mjx.put_model(model)
    mjx_data = mjx.put_data(model, data)
    jit_step = jax.jit(mjx.step)
    
    # Warmup
    mjx_data = jit_step(mjx_model, mjx_data)
    jax.block_until_ready(mjx_data.qpos)
    
    # Benchmark
    n = 10_000
    start = time.time()
    for _ in range(n):
        mjx_data = jit_step(mjx_model, mjx_data)
    jax.block_until_ready(mjx_data.qpos)
    elapsed = time.time() - start
    print(f'GPU: {n} steps in {elapsed:.2f}s ({n/elapsed:,.0f} steps/sec)')

In [ ]:
# Use dm_control's task interface for structured RL environments
env = suite.load('humanoid', 'stand')
time_step = env.reset()

print('Observation spec:')
for key, spec in env.observation_spec().items():
    print(f'  {key}: shape={spec.shape}, dtype={spec.dtype}')

print(f'\nAction spec: shape={env.action_spec().shape}')
print(f'Reward range: [{env.action_spec().minimum[0]}, {env.action_spec().maximum[0]}]')

## Next Steps

- Try other dm_control tasks: `suite.load('humanoid', 'walk')`, `suite.load('humanoid', 'run')`
- Explore `/opt/hcloud/mujoco_playground/` for advanced training examples
- Run batched MJX simulation: `python /opt/hcloud/examples/batched_humanoid_mjx.py`
- Swap in your own MJCF model: replace the xml_path and redeploy